# EmpregaData — Análise e Machine Learning
Projeto Integrador IV — Mercado de Trabalho / CAGED

Esta versão utiliza somente informação histórica disponível até o mês-base para estimar a demanda do mês seguinte.

In [ ]:
import pandas as pd
import plotly.express as px

setores=pd.read_csv('../data/historico_setores.csv',parse_dates=['date'])
ufs=pd.read_csv('../data/historico_ufs.csv',parse_dates=['date'])
setores.head()

## 1. Análise descritiva
A taxa líquida é calculada como `saldo / estoque`. Ela ajuda a comparar setores e estados de tamanhos diferentes.

In [ ]:
# Último mês disponível
ultimo=setores['date'].max()
print('Último mês:',ultimo.strftime('%m/%Y'))

# Saldo agregado por setor no último mês
s=setores[setores.date==ultimo].sort_values('Saldos',ascending=False).head(10)
px.bar(s,x='Saldos',y='setor',orientation='h',text_auto=True)

In [ ]:
# Evolução da taxa líquida dos setores
sel=setores.groupby('setor')['taxa_liquida'].mean().sort_values(ascending=False).head(5).index
p=setores[setores.setor.isin(sel)]
px.line(p,x='date',y='taxa_liquida',color='setor',markers=True)

## 2. Machine Learning
O script `src/train_model.py` cria exemplos supervisionados usando seis meses anteriores e tem como alvo o mês seguinte. O conjunto de treino termina em dezembro de 2025 e o teste usa janeiro a junho de 2026.

In [ ]:
# Treinar os modelos
!python ../src/train_model.py

In [ ]:
import json
m=json.load(open('../data/metricas_modelos.json',encoding='utf-8'))
print('Acurácia setores:',m['setor']['accuracy'])
print('Acurácia UFs:',m['uf']['accuracy'])

## 3. Interface
A interface `src/app.py` possui quatro páginas: Visão geral, Analisar área, Encontrar oportunidades e Sobre o modelo.